# NACC_DemRKL
Copied from the original project. Paths are relative to the main folder.
Original DemRKG generation and training settings are preserved.


In [ ]:
# Paths relative to the main folder; portable between Windows and Linux.
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "full_primekg.py").is_file() and (p / "paths.py").is_file())
for _path in (CLINICALKG_ROOT.parent, CLINICALKG_ROOT):
    if str(_path) in sys.path:
        sys.path.remove(str(_path))
    sys.path.insert(0, str(_path))
from clinicalKG.paths import PROJECT_ROOT, SHARED_PRIMEKG, raw_primekg_path, work_dir, asset_path
DATASET = 'NACC'
METHOD = 'DemRKL'
WORK_DIR = work_dir(DATASET, METHOD)
DATASET_DIR = PROJECT_ROOT / DATASET
if METHOD == "PrimeKGL":
    SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
# A notebook kernel may previously have imported the original project's modules.
for _name in list(sys.modules):
    if _name == "experiments" or _name.startswith("experiments."):
        _file = getattr(sys.modules[_name], "__file__", None)
        if _file and not Path(_file).resolve().is_relative_to(CLINICALKG_ROOT):
            del sys.modules[_name]
if "model1" in sys.modules:
    _file = getattr(sys.modules["model1"], "__file__", None)
    if _file and Path(_file).resolve().parent != WORK_DIR:
        del sys.modules["model1"]
if str(WORK_DIR) in sys.path:
    sys.path.remove(str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(CLINICALKG_ROOT / "results"))


In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

RAW_KG_PATH = str(raw_primekg_path())
OUTPUT_CSV_PATH = str(WORK_DIR / 'NACC_Seed_DementiaHKG.csv')

def build_seed_and_export_csv_nacc():
    print("==================================================")
    print("🕵️ 步骤 1: 加载 PrimeKG 数据库...")
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
        all_nodes = set(df['x_name'].dropna()).union(set(df['y_name'].dropna()))
        print(f"   ✅ 成功加载 {RAW_KG_PATH}，共提取独立节点 {len(all_nodes)} 个。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {RAW_KG_PATH}，请检查路径。")
        return

    print("\n==================================================")
    print("🧬 步骤 2: 初始化医学共识核心种子 (Core Seeds)...")
    
    core_seeds = {
        "Alzheimer disease": "Core_Disease", 
        "Dementia": "Core_Disease",
        "Cognitive impairment": "Core_Disease",
        "APOE": "Core_Gene",
        "MAPT": "Core_Gene",   
        "APP": "Core_Gene",    
        "PSEN1": "Core_Gene",
        "PSEN2": "Core_Gene",
        "TREM2": "Core_Gene"
    }

    print("\n==================================================")
    print("📊 步骤 3: 补全 NACC 临床与病史表型映射字典...")
    
    nacc_mapping = {
        # --- NPI-Q 精神神经症状 ---
        "npiq_DEL": "Delusions",
        "npiq_HALL": "Hallucinations",
        "npiq_AGIT": "Agitation",
        "npiq_DEPD": "Depressivity",          
        "npiq_ANX": "Anxiety",
        "npiq_ELAT": "Euphoria",
        "npiq_APA": "Apathy",
        "npiq_DISN": "Disinhibition",
        "npiq_IRR": "Irritability",
        "npiq_MOT": "Restlessness",
        "npiq_NITE": "Sleep disturbance",     
        "npiq_APP": "Poor appetite",          

        # --- 认知与功能量表 ---
        "lm_imm": "Memory impairment",     
        "lm_del": "Memory impairment",
        "craft_imm": "Memory impairment",
        "craft_del": "Memory impairment",
        "boston": "Aphasia",               
        "mint": "Aphasia",
        "animal": "Aphasia",               
        "Fwords": "Aphasia",
        "trailA": "Impaired executive functioning", 
        "trailB": "Impaired executive functioning", 
        "digitB": "Short attention span",           
        "digitBL": "Short attention span",          
        "digitF": "Short attention span",           
        "digitFL": "Short attention span",
        "numberB": "Short attention span",
        "numberBL": "Short attention span",
        "numberF": "Short attention span",
        "numberFL": "Short attention span",
        "faq_BILLS": "Impaired executive functioning", 
        "faq_TAXES": "Impaired executive functioning", 
        "faq_SHOPPING": "Memory impairment",
        "faq_REMDATES": "Memory impairment",
        "faq_GAMES": "Impaired executive functioning",
        "faq_STOVE": "Memory impairment",
        "faq_MEALPREP": "Impaired executive functioning",
        "faq_EVENTS": "Short attention span",
        "faq_PAYATTN": "Short attention span",
        "faq_TRAVEL": "Impaired executive functioning",
        "gds": "Depressivity",                      

        # --- EHR 既往病史映射 (全量精确命中版) ---
        "his_CVHATT": "myocardial infarction", 
        "his_CVAFIB": "atrial fibrillation (disease)",
        "his_CVCHF": "heart failure",
        "his_CBSTROKE": "cerebral infarction", 
        "his_CBTIA": "transient ischemic attack (disease)",
        "his_SEIZURES": "Seizure",
        "his_HYPERTEN": "hypertension",        
        "his_HYPERCHO": "Hypercholesterolemia",
        "his_DIABETES": "type 2 diabetes mellitus",
        "his_THYROID": "hypothyroidism",
        "his_B12DEF": "vitamin B12 deficiency",
        "his_PSYCDIS": "mental disorder",      
        "his_OCD": "obsessive-compulsive disorder",
        "his_ANXIETY": "Anxiety",
        "his_SCHIZ": "schizophrenia",
        "his_BIPOLAR": "bipolar disorder",
        "his_PTSD": "post-traumatic stress disorder",
        "his_ALCOHOL": "Alcoholism",
        "his_DEP2YRS": "Depressivity",
        "his_DEPOTHR": "Depressivity"          
    }

    print("\n==================================================")
    print("🔍 步骤 4: 在 PrimeKG 中校验节点有效性...")
    
    valid_records = []
    
    for seed, category in core_seeds.items():
        if seed in all_nodes:
            valid_records.append({"NACC_Feature": "Core_Prior_Knowledge", "PrimeKG_Node": seed, "Category": category})
            print(f"     ✅ 命中: {seed}")
        else:
            print(f"     ❌ 未命中: {seed}")

    for nacc_feat, pkg_node in nacc_mapping.items():
        if pkg_node in all_nodes:
            valid_records.append({"NACC_Feature": nacc_feat, "PrimeKG_Node": pkg_node, "Category": "Clinical_Mapping"})
            print(f"     ✅ 命中: {nacc_feat} -> {pkg_node}")
        else:
            print(f"     ❌ 未命中: {nacc_feat} -> {pkg_node}")

    print("\n==================================================")
    print("🚫 步骤 5: 附加全局黑名单规则...")
    for rule in ["drug", "exposure", "indication", "contraindication", "off-label use"]:
        cat = "BLACKLIST_TYPE" if rule in ["drug", "exposure"] else "BLACKLIST_REL"
        valid_records.append({"NACC_Feature": cat, "PrimeKG_Node": rule, "Category": "Rule"})

    print("\n==================================================")
    print("💾 步骤 6: 导出为 CSV 文件...")
    out_df = pd.DataFrame(valid_records)
    out_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')
    print(f"   ✅ 导出完成！文件已保存至: {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    build_seed_and_export_csv_nacc()